In [15]:
import math
import numpy as np
import matplotlib.pyplot as g
import torch
import random
from  graphviz import Digraph
%matplotlib inline

In [78]:
class Scalar:
    # Properties initialize Karnyasathi constructor (CTOR) asto 
    # CTOR is a special function
    # () = Tuple = list of numbers that you cannot change once assigned
    
    
    def __init__(self, value, label='', oprt = '', children = ()):
        # self = Jyane he function call kele ahe to/ti
        self.value = value
        self.label = label
        self.oprt  = oprt
        self.children = children
        self.der = 0.0
        self._backward = lambda: None


    def __add__(self, other):

        # If other is not "Scalar" type then convert it to Scalar type
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value + other.value, '', oprt='+', children=(self, other))

        # for back propagation
        def _backward():
            # Addition madhe
            # derivative of output will propagate to both of its operands
            # out = self + other
            # tar, out cha derivative self ani other kade propagate honar ahe
            self.der = 1.0 * out.der # 1.0 is local derivative of x in y = z + x (dy/dx = 1)
            other.der = 1.0 * out.der

        # function assignment
        out._backward = _backward

        return out

    def __radd__(self, other):
        # what is r ?
        # r = reflected (swapped operands)
        return self + other # Calls __add__

    def __rmul__(self, other): # Reflected multiplication
        return self * other

    def __pow__(self, n):
        out = Scalar(self.value ** n, label='', children=(self,), oprt=f'**{n}')

        def _backward():
            self.der = n * (self.value ** (n-1)) * out.der

        out._backward = _backward
        return out

    def __neg__(self):
        # Implement negation using mul and rmul
        return self * -1
        
    def __sub__(self, other):
        # Implement subtraction using addition + negation
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)
        
    def __mul__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value * other.value, '', oprt='*', children=(self, other))

        def _backward():
            self.der = other.value * out.der  # y = 3 * x =>  dy/dx = 3;  c = a * b then dc/da = b
            other.der = self.value * out.der

        out._backward = _backward

        return out

        # for back propagation

    def relu(self):
        # if value is negative, makes it 0
        # else keep value as-is
        out =  Scalar(0 if self.value < 0 else self.value, '', oprt = 'ReLU', children=(self,))

        def _backward():
            self.der = (out.value > 0) * out.der

        out._backward = _backward
        return out

    def backward(self):
        '''
        What should it do ?
        '''
        visited = set() # nodes from trace function
        topo = list()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.children:
                    build_topo(child)

                topo.append(v)

        build_topo(self) # jyane call kela tithun start kara

        self.der = 1.0 # dL/dL = 1.0
        for v in reversed(topo):
            v._backward()
            

In [17]:
def test_topo(root):
    visited = set() # nodes from trace function
    topo = list()
    def build_topo(v):
        if v not in visited:
            visited.add(v)
            for child in v.children:
                build_topo(child)

            topo.append(v)

    build_topo(root) # jyane call kela tithun start kara

    for v in reversed(topo):
        print(v.label)

trace(L)
        

NameError: name 'L' is not defined

In [123]:

def trace(root):
    # nodes will act as visited array
    nodes = set() # Expression Graph che nodes 
    edges = set() # Expression Graph chya edges

    def build(v):
        if v not in nodes:
            nodes.add(v)


            for child in v.children:
                edges.add((child,v))
                build(child)

    build(root)
    return nodes, edges
    
class MermaidGraph:
    """Wraps Mermaid source so Jupyter renders it inline, like a graphviz Digraph."""
    def __init__(self, source):
        self.source = source

    def _repr_mimebundle_(self, include=None, exclude=None):
        return {
            'text/vnd.mermaid': self.source,                      # JupyterLab 4.1+ / Notebook 7.1+
            'text/markdown': f"```mermaid\n{self.source}\n```",   # fallback renderer
            'text/plain': self.source,
        }

    def __str__(self):
        return self.source


def draw_mermaid(root, rankdir='LR'):
    nodes, edges = trace(root)

    def esc(s):
        return str(s).replace('"', '#quot;')   # Mermaid's escape for quotes

    lines = [f"flowchart {rankdir}"]

    for n in nodes:
        nid = f"n{id(n)}"
        lines.append(f'    {nid}["{esc(n.label)} | Val {n.value:.4f} | Der {n.der:.4f}"]')
        if n.oprt:
            lines.append(f'    {nid}_op(("{esc(n.oprt)}"))')
            lines.append(f'    {nid}_op --> {nid}')

    for n1, n2 in edges:
        lines.append(f'    n{id(n1)} --> n{id(n2)}_op')

    return MermaidGraph("\n".join(lines))


def draw_graph(root, format='svg', rankdir='LR'):
    nodes, edges = trace(root)
    
    graph = Digraph(
        format=format,
        graph_attr={'rankdir': rankdir, 'bgcolor': 'transparent'},
        node_attr={
            'style': 'filled',
            'fillcolor': '#2d2d2d',
            'color': '#d4d4d4',
            'fontcolor': '#d4d4d4',
            'fontname': 'Helvetica',
        },
        edge_attr={'color': '#d4d4d4'},
    )
    
    for n in nodes:
        graph.node(name=str(id(n)), label = "{ %s | Val %.4f | Der %.4f }" % (n.label, n.value, n.der), shape='record')
        if n.oprt:
            graph.node(name=str(id(n)) + n.oprt, label=n.oprt, fillcolor='#264f78', color='#569cd6', fontcolor='#ffffff')
            graph.edge(str(id(n)) + n.oprt, str(id(n)))
    
    for n1, n2 in edges:
        graph.edge(str(id(n1)), str(id(n2)) + n2.oprt)
    
    return graph

In [19]:
'''

# Call 
Calling a function

use of parantheses ()

add()
multiply()

# Neuron class cha CTOR call hoto
# CTOR = Jyamule Class cha instance/object tayar hoto

n = Neuron() // Neuron.__init__ = CTOR


a = Scalar (1)
b = Scalar (2)

c = a + b  => a.__add__(b) => __add__(self = a, other = b)

+  : Operator (addition)
() : Operator (function call operator)

f(x) = function of x = x is input to f

For Neuron class, we want to call / invoke each neuron with one or more inputs (Xs)

n = Neuron(3) # Neuron with 3 inputs

x = [1,2,3]

n(x) # n of x = Call Neuron n with input x
     # Use weights and biases inside this Neuron object/instance
     # and apply products + summation + activation

f(x) = x1 * w1 + x2 * w2 + ..... + bias


def fun(): # This is definition of function
    some code


fun() # function call
'''

'\n\n# Call \nCalling a function\n\nuse of parantheses ()\n\nadd()\nmultiply()\n\n# Neuron class cha CTOR call hoto\n# CTOR = Jyamule Class cha instance/object tayar hoto\n\nn = Neuron() // Neuron.__init__ = CTOR\n\n\na = Scalar (1)\nb = Scalar (2)\n\nc = a + b  => a.__add__(b) => __add__(self = a, other = b)\n\n+  : Operator (addition)\n() : Operator (function call operator)\n\nf(x) = function of x = x is input to f\n\nFor Neuron class, we want to call / invoke each neuron with one or more inputs (Xs)\n\nn = Neuron(3) # Neuron with 3 inputs\n\nx = [1,2,3]\n\nn(x) # n of x = Call Neuron n with input x\n     # Use weights and biases inside this Neuron object/instance\n     # and apply products + summation + activation\n\nf(x) = x1 * w1 + x2 * w2 + ..... + bias\n\n\ndef fun(): # This is definition of function\n    some code\n\n\nfun() # function call\n'

In [20]:
def asdf():
    a = [1,2,3,4]
    b = [1,4,9,16]
    for i,j in zip(a,b):
        print(i,j)

asdf()

1 1
2 4
3 9
4 16


In [21]:
def werwer():
    x = [1,2,3,4]
    y = 5

    z = x + [y] # means adding two lists
    print(z)

werwer()

[1, 2, 3, 4, 5]


In [22]:
[1,2,3] + 5

TypeError: can only concatenate list (not "int") to list

In [104]:
class Neuron:
    def __init__(self, n_inputs, act='relu', initial_bias=None): # CTOR
        self.weights = [Scalar(random.uniform(-1,1), label=f'w{i+1}') for i in range (n_inputs)]
        self.bias = Scalar(random.uniform(-1,1), label='b') if initial_bias == None else initial_bias
        self.act = act # relu
        self.last_pre = None # Debugging purpose
    
    # Implement function call operator "()" for Neuron object
    # so that n(x) will provide us the output of neuron n with input x
    def __call__(self, x):
        # x is of lenght 3 (x is in R^3)
        n = sum(xi * wi for xi, wi in zip(x, self.weights)) + self.bias
        
        if self.act == 'relu':
            return n.relu()

        return n # Linear

    @property # params will be accessible as a data of neuron and not a function
    def params(self):
        # concatenate two lists
        return self.weights + [self.bias]
    

In [105]:
x = [1,2,3,4]
def mysum():
    total = 0
    for element in x:
        total = total + element

        total = 0 + Scalar(1)

In [106]:
n = Neuron (2)
x = [Scalar(1.0), Scalar(2.0)]
n(x)

In [107]:
def fun():
    a = Scalar(1.0, label='a')
    b = 4 + a  # 4.__add__() because 4 is int

    c = a + 4 #=> a.__add__(1) => self =1 , other = 1 other.value , 1 is built in data type

    print(type(b))
    print(b.value)
    print(type(c))
    print(c.value)

fun()


<class '__main__.Scalar'>
5.0
<class '__main__.Scalar'>
5.0


In [108]:
print(n.params[0].label) # n.params is a data member
print(n.params[1].label)
print(n.params[2].label)

w1
w2
b


In [109]:
class Layer:
    # Layer of neurons
    # 3 , 4
    def __init__(self, n_inputs, n_outputs, **kwargs): # kwargs : keyword arguments are optional
        self.neurons = [Neuron(n_inputs, **kwargs) for _ in range(n_outputs)]

    def __call__(self, x):
        # List comprehension
        # collect output of each neuron n from list of neurons (self.neurons)
        # n(x) => Calls the Neuron object, collects the output
        outputs = [n(x) for n in self.neurons]
        return outputs[0] if len(outputs) == 1 else outputs

    @property
    def params(self):
        # Get each neuron, and each parameter in each neuron
        return [p for neuron in self.neurons for p in neuron.params]

In [110]:
class MLP:
    '''
    Multi Layer Perceptron
    '''
    def __init__(self, n_inputs, output_layers, act='relu', initial_bias=None):
        # concatenate two lists 3 [4,4,1] => [3,4,4,1]
        # sizes = [3,4,4,1]
        '''
        we need 3 layers (3 is len(output_layers))
                  0  1  2  3
        3 X 4    [3, 4, 4, 1]   => List madhle index[0] ani index[1]
        4 X 4    [3, 4, 4, 1]   => List madhle index[1] ani index[2]
        4 X 1    [3, 4, 4, 1]   => List madhle index[2] ani index[3]
                 s i z e s
        '''
        sizes = [n_inputs] + output_layers
        self.layers = [
            Layer(
                sizes[i], sizes[i+1],
                act=act,
                initial_bias=initial_bias
            )
            for i in range(len(output_layers)) 
        ]

    def __call__(self, x):
        for l in self.layers:
            x = l(x) # => calls layer l with input x

        return x

    @property
    def params(self):
        # How to calculate parameters of this MLP ?
        # l.params will call the @property decorated method of Layer l
        return [p for l in self.layers for p in l.params]
        

In [111]:
nn = MLP(3, [4,4,1]) # 3 inputs, 2 layers with 4 neurons each + 1 output

In [112]:
len(nn.params)

41

In [113]:
# x = [demand, stock, incoming] R^3
# y = max(0, demand - stock - incoming)
xs = [
    [3.0, 1.0, 1.0],   # 3 - 1 - 1 =  1
    [4.0, 1.0, 0.0],   # 4 - 1 - 0 =  3
    [2.0, 2.0, 1.0],   # 2 - 2 - 1 = -1  -> 0
    [5.0, 2.0, 1.0],   # 5 - 2 - 1 =  2
    [1.0, 0.0, 2.0],   # 1 - 0 - 2 = -1  -> 0
]
ys = [1.0, 3.0, 0.0, 2.0, 0.0]

In [114]:
type(nn)

__main__.MLP

In [116]:
random.seed(2)
nn = MLP(3, [4,4,1]) 

In [117]:
# xs =  array of arrays or array of vectors
# but neuron works on ONE VECTOR AT A TIME
ypred = [nn(x) for x in xs]

In [118]:
type(ypred)

list

In [119]:
len(ypred)

5

In [120]:
for y in ypred:
    print(y.value)

0.9966774430645892
2.5014917864710036
0.9981722528074892
3.1705611571741534
0.7023229175018586


In [121]:
loss = sum((ya - yp)**2 for ya, yp in zip(ys, ypred))

In [ ]:
draw_graph(loss)

In [125]:
loss.backward()

In [ ]:
draw_graph(loss)

In [ ]:
# Update all parameters (weights and biases) with derivatives

In [ ]:
# Again re-calculate ypred and re-calcualte loss